# 🎶 Traducción de Letras de Canciones

## Objetivo
El objetivo de este notebook es detectar el idioma de las canciones y traducir aquellas que no estén en inglés. Esto nos permitirá entrenar eficazmente un modelo de Roberta con canciones de distintos países y variedad. 🌍🎤

## Descripción del Proceso

1. **Importación de Módulos y Configuración Inicial** 📦
    - Importamos las bibliotecas necesarias como `pandas`, `langdetect`, `googletrans`, `deep_translator`, entre otras.
    - Configuramos los límites de tamaño de campo y rutas de los archivos.

2. **Carga y Filtrado de Datos** 📂
    - Cargamos los archivos CSV con las letras de canciones.
    - Filtramos las canciones que no están en inglés y aquellas que no tienen traducción.

3. **Detección de Idioma y Traducción** 🌐
    - Utilizamos `langdetect` para detectar el idioma de las letras.
    - Traducimos las letras no inglesas usando `googletrans` y `deep_translator`.

4. **Limpieza y Procesamiento de Texto** 🧹
    - Limpiamos las letras eliminando saltos de línea y espacios múltiples.
    - Dividimos textos largos para evitar errores en la traducción.

5. **Guardado de Resultados y Manejo de Errores** 💾
    - Guardamos las letras traducidas en un nuevo archivo CSV.
    - Registramos los errores encontrados durante el proceso de traducción.

6. **Verificación y Análisis** 🔍
    - Verificamos las traducciones y analizamos los resultados.
    - Fusionamos los datos traducidos con el archivo original para obtener un dataset completo.

## Herramientas Utilizadas
- `pandas` para manipulación de datos.
- `langdetect` para detección de idioma.
- `googletrans` y `deep_translator` para traducción automática.
- `tqdm` para mostrar el progreso del procesamiento.
- `transformers` para utilizar modelos avanzados de traducción como MarianMT.

## Resultados Esperados
Al final del proyecto, esperamos tener un dataset completo con letras de canciones traducidas al inglés, listo para ser utilizado en el entrenamiento de modelos de procesamiento de lenguaje natural (NLP). 🎯📊

## Cosas que se han probado
- `transformers` para utilizar modelos avanzados de traducción como MarianMT.

Facebook traductor

In [3]:
import pandas as pd

# Ruta del archivo combinado
file_path = r"C:\Users\solan\MoodTune\data\procesando\df_80-200_p7_last.csv"
output_filtered_path = r"C:\Users\solan\MoodTune\data\procesando\df_80-200_p7_to_translate.csv"

# Cargar el archivo CSV combinado
df = pd.read_csv(file_path, encoding='utf-8')

# Filtrar las filas donde la columna 'language' es nula o no es inglés
df_to_translate = df[df['language'].isnull() | (df['language'] != 'en')]

# Guardar el DataFrame filtrado en un nuevo archivo CSV
df_to_translate.to_csv(output_filtered_path, index=False, encoding='utf-8')

print(f"✅ Canciones con 'language' nulo o no en inglés guardadas en {output_filtered_path}")

C:\Users\solan\AppData\Local\Temp\ipykernel_34520\18425513.py:8: DtypeWarning: Columns (74,75,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='utf-8')


✅ Canciones con 'language' nulo o no en inglés guardadas en C:\Users\solan\MoodTune\data\procesando\df_80-200_p7_to_translate.csv


In [ ]:
import sys
import csv
import pandas as pd
from langdetect import detect
from deep_translator import GoogleTranslator
import re
from tqdm import tqdm  # Barra de progreso
import time
import os
from collections import deque  # Para mostrar solo los últimos 20 procesos

# 📌 Aumentar el límite del tamaño de campo
sys.setrecursionlimit(10000)
csv.field_size_limit(10**9)

# 📌 Configuración de archivos con los directorios correctos
PROCESANDO_DIR = "C:/Users/solan/MoodTune/data/procesando/"
PROCESADO_DIR = "C:/Users/solan/MoodTune/data/procesando/"

# 📌 Buscar el archivo más reciente en `procesando/`
file_path = os.path.join(PROCESANDO_DIR, "df_80-200_p7_to_translate.csv")

# 📌 Nombre del archivo de salida y errores
output_file_path = os.path.join(PROCESANDO_DIR, "df_80-200_p7_last_traducidas.csv")
error_file_path = os.path.join(PROCESANDO_DIR, "errors_log_traduccion.csv")

# 📌 Crear la carpeta si no existe
os.makedirs(PROCESANDO_DIR, exist_ok=True)

# 📌 Inicializar el traductor
translator = GoogleTranslator(source="auto", target="en")

# 🔹 Función para limpiar texto antes de traducir
def clean_text(text):
    if pd.isna(text):
        return ""
    return re.sub(r'\s+', ' ', text.replace('\n', ' ').strip())

# 🔹 Función para dividir textos largas (GoogleTranslator tiene un límite)
def split_text(text, max_length=500):
    return [text[i:i+max_length] for i in range(0, len(text), max_length)]

# 📌 Cargar datos originales
data = pd.read_csv(file_path, encoding='utf-8', engine='python')

# 📌 Verificar si `translated_lyrics` y `language` existen, si no, crearlas
if 'translated_lyrics' not in data.columns:
    data['translated_lyrics'] = pd.NA

if 'language' not in data.columns:
    data['language'] = pd.NA  # Crear la columna de idioma si no existe

# 📌 Crear DataFrame para errores
errors = pd.DataFrame(columns=['index', 'recording_id', 'error_message'])

# 📌 Mantener solo los últimos 20 logs en la terminal
last_logs = deque(maxlen=20)

# 🔁 Procesar y traducir
for index, row in tqdm(data.iterrows(), total=data.shape[0], desc="🔄 Traduciendo canciones"):
    try:
        # Saltar si ya está traducido
        if pd.notnull(row['translated_lyrics']) and str(row['translated_lyrics']).strip() != "":
            continue  

        lyrics_cleaned = clean_text(row['lyrics'])

        # 🔹 Evitar traducciones de textos vacíos
        if not lyrics_cleaned.strip():
            last_logs.append(f"⚠️ Letra vacía para {row['recording_id']}. Saltando...")
            continue

        # 🔹 Detectar idioma si no está en la columna "language"
        if pd.isna(row['language']) or row['language'] == "":
            try:
                lang = detect(lyrics_cleaned)
                data.at[index, 'language'] = lang  # Guardar idioma detectado
            except:
                lang = "unknown"  # Si langdetect falla, marcar como "desconocido"
        else:
            lang = row['language']

        last_logs.append(f"🧐 Procesando {index}/{len(data)} - {row['recording_id']} | Idioma detectado: {lang}")

        # 🔥 Si la canción ya está en inglés, no traducirla
        if lang == "en":
            last_logs.append(f"✅ {row['recording_id']} ya está en inglés. Saltando...")
            continue

        # 🔹 Intentar traducir hasta 3 veces si la traducción es vacía
        max_retries = 3
        translation = ""
        for attempt in range(max_retries):
            try:
                text_parts = split_text(lyrics_cleaned)
                translation = " ".join([translator.translate(part) for part in text_parts])
                if translation.strip() != "":
                    break  # Si la traducción es válida, salir del loop
            except:
                last_logs.append(f"⚠️ Intento {attempt + 1} fallido para {row['recording_id']}. Reintentando...")
                time.sleep(1)

        if translation.strip() == "":
            last_logs.append(f"❌ No se pudo traducir {row['recording_id']}. Guardando como error.")
            errors = pd.concat([errors, pd.DataFrame({'index': [index], 'recording_id': [row['recording_id']], 'error_message': ["Traducción vacía después de reintentos"]})])
            continue

        # 📌 Guardar la traducción en el DataFrame
        data.at[index, 'translated_lyrics'] = translation

        # 📌 Guardar inmediatamente en el archivo CSV para no perder datos
        pd.DataFrame({
            'recording_id': [row['recording_id']], 
            'translated_lyrics': [translation], 
            'language': [lang]  
        }).to_csv(output_file_path, mode='a', header=not os.path.exists(output_file_path), index=False, encoding='utf-8', sep=",", quoting=csv.QUOTE_MINIMAL)

        # 🔹 Mostrar progreso cada 10 traducciones
        if index % 10 == 0:
            print("\n".join(last_logs))

        time.sleep(0.2)  # Pausa para evitar bloqueos

    except Exception as e:
        errors = pd.concat([errors, pd.DataFrame({'index': [index], 'recording_id': [row['recording_id']], 'error_message': [str(e)]})])

# 📌 Guardar errores en archivo
errors.to_csv(error_file_path, index=False, encoding="utf-8")

print("\n".join(last_logs))
print(f"✅ Proceso completado. Archivo traducido guardado en {output_file_path}")

🔄 Traduciendo canciones:   0%|          | 10/7144 [00:21<4:05:26,  2.06s/it]

🧐 Procesando 0/7144 - 06150574-7d3b-409f-955a-c32b7c1dad33 | Idioma detectado: en
✅ 06150574-7d3b-409f-955a-c32b7c1dad33 ya está en inglés. Saltando...
🧐 Procesando 1/7144 - 779263a9-c234-474a-aacb-6e1ce6c9c49e | Idioma detectado: es
🧐 Procesando 2/7144 - c5ded84c-98e6-4099-bf7b-a19bf6cb7291 | Idioma detectado: fi
🧐 Procesando 3/7144 - 551aee08-e3f7-46bb-bf06-793d95f76213 | Idioma detectado: fr
🧐 Procesando 4/7144 - c105eac0-24a5-48d1-b86e-4dc0024e03e8 | Idioma detectado: it
🧐 Procesando 5/7144 - 44a5b1ba-fa9d-40dd-9c58-d39702f6457e | Idioma detectado: de
🧐 Procesando 6/7144 - cffec82e-fe60-4fbc-a36b-501e77efd21d | Idioma detectado: de
🧐 Procesando 7/7144 - 7024c111-4c20-4619-ad61-bf23b2cf14fb | Idioma detectado: fr
🧐 Procesando 8/7144 - a97b4568-4bb6-4242-a208-5720c6183e86 | Idioma detectado: es
🧐 Procesando 9/7144 - 44c7281d-a7a6-48f3-ae01-5f6832a533cc | Idioma detectado: es
🧐 Procesando 10/7144 - 70dfd89b-f7f8-4d3e-b0ab-a21b40f54c3f | Idioma detectado: fr


🔄 Traduciendo canciones:   0%|          | 20/7144 [00:50<6:07:49,  3.10s/it]

🧐 Procesando 1/7144 - 779263a9-c234-474a-aacb-6e1ce6c9c49e | Idioma detectado: es
🧐 Procesando 2/7144 - c5ded84c-98e6-4099-bf7b-a19bf6cb7291 | Idioma detectado: fi
🧐 Procesando 3/7144 - 551aee08-e3f7-46bb-bf06-793d95f76213 | Idioma detectado: fr
🧐 Procesando 4/7144 - c105eac0-24a5-48d1-b86e-4dc0024e03e8 | Idioma detectado: it
🧐 Procesando 5/7144 - 44a5b1ba-fa9d-40dd-9c58-d39702f6457e | Idioma detectado: de
🧐 Procesando 6/7144 - cffec82e-fe60-4fbc-a36b-501e77efd21d | Idioma detectado: de
🧐 Procesando 7/7144 - 7024c111-4c20-4619-ad61-bf23b2cf14fb | Idioma detectado: fr
🧐 Procesando 8/7144 - a97b4568-4bb6-4242-a208-5720c6183e86 | Idioma detectado: es
🧐 Procesando 9/7144 - 44c7281d-a7a6-48f3-ae01-5f6832a533cc | Idioma detectado: es
🧐 Procesando 10/7144 - 70dfd89b-f7f8-4d3e-b0ab-a21b40f54c3f | Idioma detectado: fr
🧐 Procesando 11/7144 - 2162fb60-3838-4cda-ac6f-0c3d259f9dad | Idioma detectado: de
🧐 Procesando 12/7144 - 496e1a36-6ad0-417e-bb5e-0ec2f006b616 | Idioma detectado: fi
🧐 Procesando 

🔄 Traduciendo canciones:   0%|          | 31/7144 [01:18<5:35:27,  2.83s/it]

...

🔄 Traduciendo canciones:   1%|          | 71/7144 [02:55<6:06:54,  3.11s/it]

🧐 Procesando 52/7144 - ccdb3994-13d3-4f9e-980b-f2f1065b12f1 | Idioma detectado: de
🧐 Procesando 53/7144 - 36b337ea-7006-4819-aa69-498250e6005e | Idioma detectado: de
🧐 Procesando 54/7144 - 33c87019-9568-4dc2-9e24-1ade4e7dcbf0 | Idioma detectado: is
🧐 Procesando 55/7144 - af6962d9-aa76-48ae-bead-a54e2cf04170 | Idioma detectado: en
✅ af6962d9-aa76-48ae-bead-a54e2cf04170 ya está en inglés. Saltando...
🧐 Procesando 56/7144 - c4ba5ed5-99b3-4f88-ae59-a524a3db71a6 | Idioma detectado: es
🧐 Procesando 57/7144 - e5df4bb3-03d3-4d56-bf27-75059431d1e0 | Idioma detectado: es
🧐 Procesando 58/7144 - dd47db9f-db1d-4456-b027-c8d20c81027f | Idioma detectado: de
🧐 Procesando 59/7144 - 4637c242-3fb8-469f-ae5b-dd5e8fd6f4b3 | Idioma detectado: es
🧐 Procesando 60/7144 - 01d5138d-2f0b-43a1-873a-72df45b8d927 | Idioma detectado: sv
🧐 Procesando 61/7144 - b8f8d034-069f-4250-9dc0-bae2ff219616 | Idioma detectado: fr
🧐 Procesando 62/7144 - f1504df9-6c3f-43cc-8eb8-6ae24edfc311 | Idioma detectado: pl
🧐 Procesando 63/7

🔄 Traduciendo canciones:   1%|          | 80/7144 [03:35<14:16:27,  7.27s/it]

🧐 Procesando 65/7144 - aebc6b1e-4c3a-4fee-8036-2f6ffe1335cb | Idioma detectado: it
🧐 Procesando 66/7144 - 56e19114-f7a8-45a6-b938-3f5503b381b0 | Idioma detectado: pl
🧐 Procesando 67/7144 - 66c257f7-54ce-4d3d-a852-7d5f1f1e181c | Idioma detectado: es
🧐 Procesando 68/7144 - 5426e97b-b93d-4257-824b-bf953c2ecbdd | Idioma detectado: es
🧐 Procesando 69/7144 - 3c2d34b3-c608-44a4-84b4-ad174552255a | Idioma detectado: fr
🧐 Procesando 70/7144 - 775751e3-f0f4-4068-81ec-187cdc6dbd54 | Idioma detectado: fi
🧐 Procesando 71/7144 - d318dde6-7847-4759-8aec-970da95ed15b | Idioma detectado: es
🧐 Procesando 72/7144 - dcc6f354-714a-46b3-a5ac-12e9c40a5ebb | Idioma detectado: de
🧐 Procesando 73/7144 - af64188e-b531-4a1e-9f1c-7794ce57f40e | Idioma detectado: it
🧐 Procesando 74/7144 - 86bc1022-f4ab-4b79-879b-c466dd42959c | Idioma detectado: it
🧐 Procesando 75/7144 - d7de2557-ab34-410a-89ce-428012f9ef27 | Idioma detectado: es
🧐 Procesando 76/7144 - 16992dfe-b850-4889-9e4f-93664668b243 | Idioma detectado: pt
🧐 Pr

🔄 Traduciendo canciones:   1%|          | 87/7144 [03:55<5:18:22,  2.71s/it] 

In [1]:
import sys
import csv
import pandas as pd
from langdetect import detect
from deep_translator import GoogleTranslator
import re
from tqdm import tqdm  # Barra de progreso
import time
import os
from collections import deque  # Para mostrar solo los últimos 20 procesos

# 📌 Aumentar el límite del tamaño de campo
sys.setrecursionlimit(10000)
csv.field_size_limit(10**9)

# 📌 Configuración de archivos con los directorios correctos
PROCESANDO_DIR = "C:/Users/solan/MoodTune/data/procesando/"
PROCESADO_DIR = "C:/Users/solan/MoodTune/data/procesado/"

# 📌 Buscar el archivo más reciente en `procesando/`
files = [f for f in os.listdir(PROCESANDO_DIR) if f.startswith("df_80-200_p1_with-lyrics") and f.endswith(".csv")]
if not files:
    raise FileNotFoundError("No se encontró ningún archivo en la carpeta 'procesando/' que comience con '0_for_last_'.")

files.sort(reverse=True)  # Ordenar por nombre (fecha y hora)
file_path = os.path.join(PROCESANDO_DIR, files[0])  # Seleccionar el más reciente

# 📌 Nombre del archivo de salida y errores
output_file_path = os.path.join(PROCESANDO_DIR, "df_80-200_p1_traducidas.csv")
error_file_path = os.path.join(PROCESANDO_DIR, "errors_log_traduccion.csv")

# 📌 Crear la carpeta si no existe
os.makedirs(PROCESANDO_DIR, exist_ok=True)

# 📌 Inicializar el traductor
translator = GoogleTranslator(source="auto", target="en")

# 🔹 Función para limpiar texto antes de traducir
def clean_text(text):
    if pd.isna(text):
        return ""
    return re.sub(r'\s+', ' ', text.replace('\n', ' ').strip())

# 🔹 Función para dividir textos largas (GoogleTranslator tiene un límite)
def split_text(text, max_length=500):
    return [text[i:i+max_length] for i in range(0, len(text), max_length)]

# 📌 Cargar datos originales
data = pd.read_csv(file_path, encoding='utf-8', engine='python')

# 📌 Verificar si `translated_lyrics` y `language` existen, si no, crearlas
if 'translated_lyrics' not in data.columns:
    data['translated_lyrics'] = pd.NA

if 'language' not in data.columns:
    data['language'] = pd.NA  # Crear la columna de idioma si no existe

# 📌 Crear DataFrame para errores
errors = pd.DataFrame(columns=['index', 'recording_id', 'error_message'])

# 📌 Mantener solo los últimos 20 logs en la terminal
last_logs = deque(maxlen=20)

# 🔁 Procesar y traducir
for index, row in tqdm(data.iterrows(), total=data.shape[0], desc="🔄 Traduciendo canciones"):
    try:
        # Saltar si ya está traducido
        if pd.notnull(row['translated_lyrics']) and str(row['translated_lyrics']).strip() != "":
            continue  

        lyrics_cleaned = clean_text(row['lyrics'])

        # 🔹 Evitar traducciones de textos vacíos
        if not lyrics_cleaned.strip():
            last_logs.append(f"⚠️ Letra vacía para {row['recording_id']}. Saltando...")
            continue

        # 🔹 Detectar idioma si no está en la columna "language"
        if pd.isna(row['language']) or row['language'] == "":
            try:
                lang = detect(lyrics_cleaned)
                data.at[index, 'language'] = lang  # Guardar idioma detectado
            except:
                lang = "unknown"  # Si langdetect falla, marcar como "desconocido"
        else:
            lang = row['language']

        last_logs.append(f"🧐 Procesando {index}/{len(data)} - {row['recording_id']} | Idioma detectado: {lang}")

        # 🔥 Si la canción ya está en inglés, no traducirla
        if lang == "en":
            last_logs.append(f"✅ {row['recording_id']} ya está en inglés. Saltando...")
            continue

        # 🔹 Intentar traducir hasta 3 veces si la traducción es vacía
        max_retries = 3
        translation = ""
        for attempt in range(max_retries):
            try:
                text_parts = split_text(lyrics_cleaned)
                translation = " ".join([translator.translate(part) for part in text_parts])
                if translation.strip() != "":
                    break  # Si la traducción es válida, salir del loop
            except:
                last_logs.append(f"⚠️ Intento {attempt + 1} fallido para {row['recording_id']}. Reintentando...")
                time.sleep(1)

        if translation.strip() == "":
            last_logs.append(f"❌ No se pudo traducir {row['recording_id']}. Guardando como error.")
            errors = pd.concat([errors, pd.DataFrame({'index': [index], 'recording_id': [row['recording_id']], 'error_message': ["Traducción vacía después de reintentos"]})])
            continue

        # 📌 Guardar la traducción en el DataFrame
        data.at[index, 'translated_lyrics'] = translation

        # 📌 Guardar inmediatamente en el archivo CSV para no perder datos
        pd.DataFrame({
            'recording_id': [row['recording_id']], 
            'translated_lyrics': [translation], 
            'language': [lang]  
        }).to_csv(output_file_path, mode='a', header=not os.path.exists(output_file_path), index=False, encoding='utf-8', sep=",", quoting=csv.QUOTE_MINIMAL)

        # 🔹 Mostrar progreso cada 10 traducciones
        if index % 10 == 0:
            print("\n".join(last_logs))

        time.sleep(0.2)  # Pausa para evitar bloqueos

    except Exception as e:
        errors = pd.concat([errors, pd.DataFrame({'index': [index], 'recording_id': [row['recording_id']], 'error_message': [str(e)]})])

# 📌 Guardar errores en archivo
errors.to_csv(error_file_path, index=False, encoding="utf-8")

print("\n".join(last_logs))
print(f"✅ Proceso completado. Archivo traducido guardado en {output_file_path}") 


🔄 Traduciendo canciones:   2%|▏         | 351/17048 [00:28<33:03,  8.42it/s]

✅ 26120a3b-7c94-4229-877c-0ce536a4f8e4 ya está en inglés. Saltando...
🧐 Procesando 341/17048 - 7150fc7e-2f09-486a-b9c3-0aae34276b65 | Idioma detectado: en
✅ 7150fc7e-2f09-486a-b9c3-0aae34276b65 ya está en inglés. Saltando...
🧐 Procesando 342/17048 - fa0b9443-cbda-4c61-aa1e-74ece0772e06 | Idioma detectado: en
✅ fa0b9443-cbda-4c61-aa1e-74ece0772e06 ya está en inglés. Saltando...
🧐 Procesando 343/17048 - 443e75ce-7d1f-4190-b584-658d3900a8de | Idioma detectado: en
✅ 443e75ce-7d1f-4190-b584-658d3900a8de ya está en inglés. Saltando...
🧐 Procesando 344/17048 - 6a32e7b6-306e-4623-ab93-82eeecf298d9 | Idioma detectado: en
✅ 6a32e7b6-306e-4623-ab93-82eeecf298d9 ya está en inglés. Saltando...
🧐 Procesando 345/17048 - 6a258ee9-6e2e-4c99-94a2-03e418b942cd | Idioma detectado: en
✅ 6a258ee9-6e2e-4c99-94a2-03e418b942cd ya está en inglés. Saltando...
🧐 Procesando 346/17048 - e044c3af-7b28-4dce-b904-e0e6cbf91e0c | Idioma detectado: en
✅ e044c3af-7b28-4dce-b904-e0e6cbf91e0c ya está en inglés. Saltando...


🔄 Traduciendo canciones:   2%|▏         | 381/17048 [00:43<1:09:41,  3.99it/s]

🧐 Procesando 370/17048 - c10cfef0-69b0-4834-ba92-3e95e4c6f28b | Idioma detectado: en
✅ c10cfef0-69b0-4834-ba92-3e95e4c6f28b ya está en inglés. Saltando...
🧐 Procesando 371/17048 - 53733f2e-12bb-4d0a-8dfc-c58d93e0060d | Idioma detectado: fr
🧐 Procesando 372/17048 - 2830f016-9968-432d-b6d9-83f0fbac135f | Idioma detectado: en
✅ 2830f016-9968-432d-b6d9-83f0fbac135f ya está en inglés. Saltando...
🧐 Procesando 373/17048 - 06128fee-a368-44b5-b061-c679a3f77fd3 | Idioma detectado: en
✅ 06128fee-a368-44b5-b061-c679a3f77fd3 ya está en inglés. Saltando...
🧐 Procesando 374/17048 - c6ec7923-fae4-4d68-b906-37e484115c0a | Idioma detectado: en
✅ c6ec7923-fae4-4d68-b906-37e484115c0a ya está en inglés. Saltando...
🧐 Procesando 375/17048 - e01b91ea-3967-4444-9573-ff8f5b3d8726 | Idioma detectado: en
✅ e01b91ea-3967-4444-9573-ff8f5b3d8726 ya está en inglés. Saltando...
🧐 Procesando 376/17048 - 1c751b9f-111c-4412-91e7-9b95ad1f6216 | Idioma detectado: en
✅ 1c751b9f-111c-4412-91e7-9b95ad1f6216 ya está en inglé

🔄 Traduciendo canciones:   9%|▉         | 1531/17048 [00:47<02:24, 107.21it/s]

...

🧐 Procesando 16730/17048 - a0003c83-bda6-4a15-8c0c-52f265d725cc | Idioma detectado: en
✅ a0003c83-bda6-4a15-8c0c-52f265d725cc ya está en inglés. Saltando...
🧐 Procesando 16731/17048 - a1d48def-1a31-4f15-9453-385d471cdf1b | Idioma detectado: en
✅ a1d48def-1a31-4f15-9453-385d471cdf1b ya está en inglés. Saltando...
🧐 Procesando 16732/17048 - 54f3b888-18de-4ddb-b4af-07b2a5ab8f3b | Idioma detectado: en
✅ 54f3b888-18de-4ddb-b4af-07b2a5ab8f3b ya está en inglés. Saltando...
🧐 Procesando 16733/17048 - f6d85b27-2f32-4ea4-8941-673ef8a8b8e5 | Idioma detectado: en
✅ f6d85b27-2f32-4ea4-8941-673ef8a8b8e5 ya está en inglés. Saltando...
🧐 Procesando 16734/17048 - 57ed0c3c-aaa4-4001-9790-04435edfd16a | Idioma detectado: en
✅ 57ed0c3c-aaa4-4001-9790-04435edfd16a ya está en inglés. Saltando...
🧐 Procesando 16735/17048 - 6c5d5819-47c9-4fdb-9994-735d1822000d | Idioma detectado: en
✅ 6c5d5819-47c9-4fdb-9994-735d1822000d ya está en inglés. Saltando...
🧐 Procesando 16736/17048 - 47a2a1f9-0379-455b-9c29-57a02dd

🔄 Traduciendo canciones:  99%|█████████▉| 16845/17048 [1:36:48<00:28,  7.23it/s]

✅ f6efef64-a44f-4b75-918d-4b9c420442a6 ya está en inglés. Saltando...
🧐 Procesando 16861/17048 - 63f95908-678f-427e-997b-20312e41791f | Idioma detectado: en
✅ 63f95908-678f-427e-997b-20312e41791f ya está en inglés. Saltando...
🧐 Procesando 16862/17048 - 6e164d7a-d3dd-4a7b-950f-f9ad424000ab | Idioma detectado: en
✅ 6e164d7a-d3dd-4a7b-950f-f9ad424000ab ya está en inglés. Saltando...
🧐 Procesando 16863/17048 - d3908498-597f-4cb0-9c36-1eb82fa6fbeb | Idioma detectado: en
✅ d3908498-597f-4cb0-9c36-1eb82fa6fbeb ya está en inglés. Saltando...
🧐 Procesando 16864/17048 - 953a2da3-982f-47b8-b138-4594b2d46991 | Idioma detectado: en
✅ 953a2da3-982f-47b8-b138-4594b2d46991 ya está en inglés. Saltando...
🧐 Procesando 16865/17048 - 0d27b9af-72d6-4be9-a314-f747e52c8892 | Idioma detectado: en
✅ 0d27b9af-72d6-4be9-a314-f747e52c8892 ya está en inglés. Saltando...
🧐 Procesando 16866/17048 - d28ad6bc-964b-491e-99a9-4132e053f5bb | Idioma detectado: en
✅ d28ad6bc-964b-491e-99a9-4132e053f5bb ya está en inglés. 

🔄 Traduciendo canciones: 100%|██████████| 17048/17048 [1:37:03<00:00,  2.93it/s]

✅ bb4d276f-64b0-426b-9a53-2542ebd7c1c4 ya está en inglés. Saltando...
🧐 Procesando 17038/17048 - 2fedcd61-5239-4db8-a928-1fb7ca2d60ec | Idioma detectado: en
✅ 2fedcd61-5239-4db8-a928-1fb7ca2d60ec ya está en inglés. Saltando...
🧐 Procesando 17039/17048 - 31bcf421-cb78-44c4-9929-91e3ff935496 | Idioma detectado: en
✅ 31bcf421-cb78-44c4-9929-91e3ff935496 ya está en inglés. Saltando...
🧐 Procesando 17040/17048 - c197e9a9-7ee1-47ce-98e4-560b6e089213 | Idioma detectado: en
✅ c197e9a9-7ee1-47ce-98e4-560b6e089213 ya está en inglés. Saltando...
🧐 Procesando 17041/17048 - ea2aaf32-448e-4485-95eb-7462f8fd06c5 | Idioma detectado: en
✅ ea2aaf32-448e-4485-95eb-7462f8fd06c5 ya está en inglés. Saltando...
🧐 Procesando 17042/17048 - 6e29798a-8363-4af4-8b20-1362b76c804f | Idioma detectado: en
✅ 6e29798a-8363-4af4-8b20-1362b76c804f ya está en inglés. Saltando...
🧐 Procesando 17043/17048 - 0a179d06-4747-4360-800a-8475320ca3d7 | Idioma detectado: it
🧐 Procesando 17044/17048 - 7426d47a-d3a1-4b85-becd-2d31414

In [28]:
import pandas as pd
import os
import re

# Rutas de los archivos
original_file_path = r"C:\Users\solan\MoodTune\data\procesando\df_80-200_p1_with-lyrics.csv"
translated_file_path = r"C:\Users\solan\MoodTune\data\procesando\df_80-200_p1_traducidas.csv"
output_merged_file_path = r"C:\Users\solan\MoodTune\data\procesado\df_80-200_p1_traducidas_1.csv"

# Cargar datos originales y traducidos
original_data = pd.read_csv(original_file_path, encoding='utf-8')
translated_data = pd.read_csv(translated_file_path, encoding='utf-8')

# Realizar la unión de los DataFrames en base al recording_id
merged_data = pd.merge(original_data, translated_data[['recording_id', 'translated_lyrics', 'language']], on='recording_id', how='left')

# Actualizar las columnas 'translated_lyrics' y 'language' en el DataFrame original con los valores del DataFrame traducido
merged_data['translated_lyrics'] = merged_data['translated_lyrics_x'].fillna(merged_data['translated_lyrics_y'])
merged_data['language'] = merged_data['language_x'].fillna(merged_data['language_y'])

# Limpiar la columna 'lyrics' para eliminar solo los saltos de línea
merged_data['lyrics'] = merged_data['lyrics'].apply(lambda x: re.sub(r'\n+', ' ', str(x).strip()) if pd.notna(x) else x)

# Eliminar las columnas temporales 'translated_lyrics_x', 'translated_lyrics_y', 'language_x' y 'language_y'
merged_data.drop(columns=['translated_lyrics_x', 'translated_lyrics_y', 'language_x', 'language_y'], inplace=True)

# Guardar el resultado en un nuevo archivo CSV
merged_data.to_csv(output_merged_file_path, index=False, encoding='utf-8')

print(f"✅ Dataset combinado guardado en {output_merged_file_path}")

KeyError: 'translated_lyrics_x'

In [10]:
#ver hasta 200 letras de cada columna y fila completa
pd.set_option('display.max_colwidth', 200)


In [78]:
#ver si hay duplicados según recording_id en df
duplicates = df[df.duplicated(subset=['recording_id'], keep=False)]
print(f"🔍 Duplicados encontrados: {duplicates.shape[0]}")
duplicates
# eliminar duplicados
df = df.drop_duplicates(subset=['recording_id'], keep='first')
print(f"✅ Duplicados eliminados. Número de filas actual: {df.shape[0]}")

🔍 Duplicados encontrados: 2
✅ Duplicados eliminados. Número de filas actual: 12630


In [ ]:
# ver donde spotify url es nulo o vacío
missing_urls = df[df['spotify_url'].isnull() | (df['spotify_url'] == '')]
print(f"🔍 URLs de Spotify faltantes: {missing_urls.shape[0]}",
      missing_urls)
# eliminar filas con URLs faltantes
df = df.dropna(subset=['spotify_url'])
# guardar el archivo actualizado como 0_para_listas.csv
df.to_csv('C:\\Users\\solan\\MoodTune\\data\\procesando\\0_para_listas.csv', index=False, encoding='utf-8')


print(f"✅ Filas eliminadas con URLs faltantes. Número de filas actual: {df.shape[0]}")

🔍 URLs de Spotify faltantes: 0 Empty DataFrame
Columns: [artist_name, song_name, recording_id, danceable, not_danceable, male, female, timbre_bright, timbre_dark, tonal, atonal, instrumental, voice, dortmund_alternative, dortmund_blues, dortmund_electronic, dortmund_folkcountry, dortmund_funksoulrnb, dortmund_jazz, dortmund_pop, dortmund_raphiphop, dortmund_rock, electronic_ambient, electronic_dnb, electronic_house, electronic_techno, electronic_trance, rosamerica_cla, rosamerica_dan, rosamerica_hip, rosamerica_jaz, rosamerica_pop, rosamerica_rhy, rosamerica_roc, rosamerica_spe, tzanetakis_blu, tzanetakis_cla, tzanetakis_cou, tzanetakis_dis, tzanetakis_hip, tzanetakis_jaz, tzanetakis_met, tzanetakis_pop, tzanetakis_reg, tzanetakis_roc, ismir04_rhythm_ChaChaCha, ismir04_rhythm_Jive, ismir04_rhythm_Quickstep, ismir04_rhythm_Rumba-American, ismir04_rhythm_Rumba-International, ismir04_rhythm_Rumba-Misc, ismir04_rhythm_Samba, ismir04_rhythm_Tango, ismir04_rhythm_VienneseWaltz, ismir04_rhyth

In [74]:
# ejemplos con nombre artista, nombre canción, url spotify letra, letra traducida 
file = r"C:\Users\solan\MoodTune\data\procesado\0_spoti_last_traduc_filtered.csv"
df = pd.read_csv(file)
# ejemplos con nombre artista, nombre canción, url spotify letra, letra traducida
df[['artist_name', 'song_name', 'language', 'spotify_url', 'lyrics', 'translated_lyrics']].sample(10)

,artist_name,song_name,language,spotify_url,lyrics,translated_lyrics
9439,lyfe jennings,rock,en,https://open.spotify.com/track/4JHEq51TdSPjC4hXRRCppq,me channel marvin gaye on this one the universe baby pick a star any star baby bought myself a oneway flight to the nearest star traveling at the speed of light to get where you are ive been trave...,NaN
5736,austin wintory,underground,en,https://open.spotify.com/track/0fKWz3HVqBjWztAKI9UYRg,take a look round lively old london buzzing crowds we sweat and we revel redcheeked shouts and songs in the flicker of the gaslight eager blighty bursts from the cobblestones racing climbing bloom...,NaN
59,gwydion,triskelion horde is nigh,en,https://open.spotify.com/track/5qhhDaAig16oHwN837YIcV,three horns triskelion welcome moving mass of seasoned men that face rain and the wildest storms stoically with the same commotion of squashing a pile of worms we travel for miles miles yards not ...,NaN
2003,jeezy,i got what it takes,en,https://open.spotify.com/track/5NQRCj8bbpMYOgmYrfH77J,its hard out here for a pimp but u already rap a lil trap a lil so i hustle and flow everyday im hustlin tryna meet ma quota nextel chirpin said hey need dat quota 42 hundred look i got that fo ya...,NaN
5892,chris robinson brotherhood,meanwhile in the gods...,en,https://open.spotify.com/track/4O71PielTlrzZDVENrWBvi,older than the mountain younger that this day barefoot dreaming sleeping children dance a figure 8 lost soft velvet in a state of decay sweet tooth youth could take the abuse but not stand the cag...,NaN
7873,teleman,monday morning,en,https://open.spotify.com/track/1Nj57FJwrWgbCmlp8Hx9jF,verse 1 i dont care if its saturday night or monday morning under the plastic sky i fell asleep with the radio on i started dreaming you were in every song verse 2 and i dont care if you were yell...,NaN
8198,nielson,hoogste versnelling,nl,https://open.spotify.com/track/264g6cOU5uNfXLMj0QOnU4,intro hey hey hey hey chorus dus geloof in jezelf juist wanneer het lijkt alsof alsof het niet meer loopt je kan rennen rennen rennen met je hoofd omhoog hoog hoogste versnelling wordt wakker want...,intro hey hey hey hey chorus believe in yourself precisely when it looks like it is no longer running you can run with your head up with your head up high highest gear awake because this is your d...
7196,looptroop rockers,professional dreamers,en,https://open.spotify.com/track/72o9g0eQ81LlHE2txjisok,living the dream living living the dream dont wanna wake up verse 1 promoe i never dreamed about getting paid i dreamed that the people never forget the name looptroop i dreamed about getting fame...,NaN
260,dead by april,let the world know,en,https://open.spotify.com/track/3AVhvkX5a1VDcsMmNevi9P,trust the words i say im willing to go the distance but im a little bit fragile im just saying lets take this slow dont get me wrong let em know write it in magazines in the papers put it all over...,NaN
6108,cambriana,face to face,en,https://open.spotify.com/track/4AWDoGEhA1UnECtyDP0Bo6,wendy youre stuck in my game i know wouldnt be here without pain for sure 50 years on no one will know the delicate humdrum ways of this love if you can keep up someday finally face to face its so...,NaN


In [67]:
import pandas as pd

# Ruta del archivo
file_path = r"C:\Users\solan\MoodTune\data\procesado\0_spoti_last_traduc2.csv"

# Cargar el archivo CSV
df = pd.read_csv(file_path, encoding='utf-8')

# Seleccionar las filas que quiero eliminar según el recording_id y eliminarlas
recording_ids_to_remove = [
    'f35eec7b-8f08-4f6d-bed1-a67c4b9b0d4c', '9979b5a4-1a5d-4b6f-9004-234bbc82dc13', '4969c286-4ece-4c2b-96e6-df3527feba8a',
    '1faabf48-0822-4680-9678-2968f3de1a73', '914d3c88-d8df-4cdd-a6b2-2a610d2b6576', '645eaf75-1464-4e1d-974e-5b1c7391e710',
    '2e47fae9-38cb-41f0-967f-d06c5f4384af', '8cfcecc0-cac8-4dbb-ac06-2b3436b59a7f', 'eecf8953-eec0-4bf3-8245-b55e44a74dab',
    'e9b488c3-3134-464e-b163-79ffab430568', 'c774abd3-7323-4614-8a80-173ada2017ab', 'dcd8e392-9fe8-4e6e-9587-61f39d26faa2',
    '720dffee-082d-4624-ba14-c906794da152', '4608246e-012d-4f00-9182-6a239c014aa5', '8d3a78bf-db98-4476-aabf-326cf4bfb0c4',
    'afdc83d0-a815-4ce5-bb88-5ddb32886594', '75a3afb0-5cbe-47a1-90c9-086aadb432c6', '06e6e3d9-39d2-4c3d-97c4-d8ca3a44f08c',
    'ab5d00d4-d01e-4277-9bce-7bf9824225b1', '15683acf-b5ad-4502-939b-9cb3cae7b50e', '0345d4d6-a39d-47bf-acd4-3a5f4a315fd5',
    '65850282-7ec2-488a-935c-fb319f43da50', '517b5d24-9e13-4560-9d07-6237608bfe78'
]
df = df[~df['recording_id'].isin(recording_ids_to_remove)]

# Rellenar las filas restantes con valores nulos en la columna 'language' con 'en'
df['language'].fillna('en', inplace=True)

# Guardar el DataFrame actualizado en un nuevo archivo CSV
output_file_path = r"C:\Users\solan\MoodTune\data\procesado\0_spoti_last_traduc_filtered.csv"
df.to_csv(output_file_path, index=False, encoding='utf-8')

print(f"✅ Filas eliminadas y archivo guardado en {output_file_path}")

C:\Users\solan\AppData\Local\Temp\ipykernel_22936\293487216.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['language'].fillna('en', inplace=True)


✅ Filas eliminadas y archivo guardado en C:\Users\solan\MoodTune\data\procesado\0_spoti_last_traduc_filtered.csv


In [23]:
# ver todo el output en la consola de la celda
pd.set_option('display.max_rows', None)

In [71]:
#ver columnas únicas de df  y contar valores únicos viendo todas las filas

df.nunique()

artist_name                            6232
song_name                             11791
recording_id                          12630
danceable                               988
not_danceable                           988
male                                    988
female                                  988
timbre_bright                           987
timbre_dark                             987
tonal                                   989
atonal                                  989
instrumental                            986
voice                                   986
dortmund_alternative                    487
dortmund_blues                          388
dortmund_electronic                     996
dortmund_folkcountry                    621
dortmund_funksoulrnb                     69
dortmund_jazz                           244
dortmund_pop                            124
dortmund_raphiphop                      199
dortmund_rock                           519
electronic_ambient              

In [72]:
# valores únicos de language
df['language'].unique()

array(['en', 'es', 'haw', 'ro', 'fi', 'de', 'fr', 'bn', 'fil', 'sv', 'la',
       'pt', 'it', 'ko', 'ca', 'so', 'hr', 'nl', 'da', 'ja', 'cy', 'ru',
       'ig', 'no', 'hi', 'ga', 'ht', 'jv', 'af', 'ms', 'fa', 'zh', 'id',
       'pl', 'ne', 'et', 'eo', 'az', 'sl', 'mg', 'tr', 'sw', 'is', 'sk',
       'co', 'yo', 'zu', 'tl', 'ha', 'hu', 'gl', 'lt', 'eu', 'mt', 'sm',
       'mi', 'th'], dtype=object)

In [70]:
#contar nulos en language
df['language'].isnull().sum()

np.int64(0)

In [69]:
# ver todas las columnas con nulos en language donde se vea 'artist_name', 'song_name', 'language', 'lyrics' y spotify url    
df[df['language'].isnull()][['artist_name', 'song_name', 'recording_id' , 'language', 'lyrics', 'spotify_url']]

,artist_name,song_name,recording_id,language,lyrics,spotify_url


In [68]:
#ver ejemplos donde language es nulo y que se vea la letra  
# Filtrar las filas donde la columna 'language' es nula
# y mostrar las columnas 'artist_name', 'song_name', 'language', 'lyrics'   y canciones traducidas
# de las primeras filas
df[df['language'].isnull()][['artist_name', 'song_name', 'language', 'lyrics', 'translated_lyrics']].sample(5)

ValueError: a must be greater than 0 unless no samples are taken

In [37]:
#nulos idioma  en df
df[df['language'].isnull()]

,artist_name,song_name,recording_id,danceable,not_danceable,male,female,timbre_bright,timbre_dark,tonal,...,spotify_url,album_name,album_release_date,duration_ms,popularity,genres,tags,views,translated_lyrics,language
23,jb and the moonshine band,doin fine,c6804ef5-1496-4768-8e11-508b4bb04222,0.246,0.754,0.003,0.997,0.984,0.016,0.973,...,https://open.spotify.com/track/6HGN7nA0O2u63uRGn1AjfT,Jb and The Moonshine Band,2010-07-12,216097.0,2.0,"beer, kid rock, scat, racism, trump",Unknown,NaN,NaN,NaN
49,the limousines,triangle circle square,8e48e54c-3e8c-4a06-ba0c-e438e0cd00e8,0.930,0.070,0.997,0.003,0.161,0.839,0.755,...,https://open.spotify.com/track/4XNYWcyRl5helntSSJWlQr,Get Sharp,2011-04-26,156466.0,5.0,"indie, electronic, experimental, electronica, electropop","alternative rock, alternative pop, New Discoveries, groovunky",NaN,NaN,NaN
143,digitalism,no cash,5c3b3dc0-41ea-490d-a993-cb6608e0e19d,0.967,0.033,0.479,0.521,0.951,0.049,0.730,...,https://open.spotify.com/track/4XqvBvciliAIcxpi9vpAZi,Mirage,2016-05-13,350320.0,10.0,"electronic, electro, dance, seen live, german",Unknown,NaN,NaN,NaN
390,hanzel und gretyl,let the planets burn,83cb4e73-7d69-41dd-9751-211dbdab4c7c,1.000,0.000,0.619,0.381,0.098,0.902,0.258,...,https://open.spotify.com/track/4bA1PXPjsvoXVYyMyYUDpH,Uber Alles,2003-05-20,204173.0,9.0,"industrial metal, industrial, industrial rock, german, metal","industrial, industrial metal, metal, 9th evolution, heavy metal",NaN,NaN,NaN
421,mavado,cleara,cd3b2720-4be8-4613-9286-179fb8666759,0.985,0.015,0.414,0.586,0.835,0.165,0.005,...,https://open.spotify.com/track/4okA5cv1Q4gkLFt7KjmAe7,Cleara,2011-01-11,142666.0,4.0,"dancehall, reggae, gangsta, jamaican, Bashment","dancehall, mavado, gully god",NaN,NaN,NaN
668,ben caplan,i got me a woman,d3367565-c71a-4f77-9093-35274c79033a,0.992,0.008,0.183,0.817,0.132,0.868,0.906,...,https://open.spotify.com/track/6C5Vd7aQgpbTIVh3JGBJCg,Birds With Broken Wings,2015-09-18,186186.0,24.0,"seen live, folk, blues, Canadian, beard",Unknown,NaN,NaN,NaN
787,they might be giants,and mom and kid,5ea4c1ba-ee53-41a4-add4-9ad257fa22d8,0.919,0.081,0.171,0.829,0.976,0.024,0.409,...,https://open.spotify.com/track/6nn4RbxQa3LAllA9QBo57b,Why?,2015-11-27,82810.0,2.0,"alternative, indie, rock, seen live, geek rock",Unknown,NaN,NaN,NaN
812,chelsea wolfe,welt,c0dd0962-b882-40c8-a02b-fd5386294668,0.594,0.406,0.962,0.038,0.000,1.000,0.312,...,https://open.spotify.com/track/0NMjFAr3aJf4z4vIeFz6Vs,Hiss Spun,2017-09-22,114443.0,19.0,"experimental, folk, female vocalists, singer-songwriter, Gothic","noise, female vocalists, dark, drone, Gothic Rock",NaN,NaN,NaN
1107,rozwell kid,my saturn,a3394492-d770-482c-a539-4a329b8408f0,0.428,0.572,0.549,0.451,0.284,0.716,0.314,...,https://open.spotify.com/track/6Yx2lEL1L2Wgm57GBY5wIk,The Rozwell Kid LP,2011-09-06,180671.0,2.0,"alternative rock, power pop, indie rock, seen live, indie",Unknown,NaN,NaN,NaN
1118,fehrplay,everywhere you go,ba1ffecf-a3f6-443c-98bb-918ba4bed8f4,0.997,0.003,0.687,0.313,0.033,0.967,0.023,...,https://open.spotify.com/track/5qAr6loIEuuDMK25Lsjymq,Everywhere You Go (Radio Edit),2014-07-15,205740.0,13.0,"electronic, Progressive House, House, norwegian, house beach electronic",trance,NaN,NaN,NaN


In [ ]:
df_non_english.sample(20)

,artist_name,song_name,spotify_url,language,lyrics,translated_lyrics
61496,yelle,le grand saut,https://open.spotify.com/track/6ro7kKnnbgMVllC2ij5VCL,fr,cours ne te retourne pas le futur ouvre grand ses bras jour j pour faire le bon choix je monte a bord du vaisseau aujourdhui le grand saut cours ne te retourne pas le futur ouvre grand ses bras jour j pour faire le bon choix je monte a bord du vaisseau aujourdhui le grand saut le hasard sorganise et il suivra mes pas je mimmunise la pluie ne tombe pas sur moi mon hoodie en maille de fer lui resistera en un eclair je lui ouvre la voie le hasard sorganise et il suivra mes pas je mimmunise la pluie ne tombe pas sur moi mon hoodie en maille de fer lui resistera en un eclair je lui ouvre la voie cours ne te retourne pas le futur ouvre grand ses bras jour j pour faire le bon choix je monte a bord du vaisseau aujourdhui le grand saut le grand saut,lessons do not turn you around the future opens big day J to make the right choice I go up on board the ship today the big leap course does not turn you around the future opens big day D to make the right choice I go up on boardFrom the vessel today the big jump the sorganized chance and he will follow my steps I mimminate the rain does not fall on me my hoodie in iron mesh will resist him in an enclosure I opens the way to him the sorganized chance and he will follow my steps I mimmise theRain does not fall on me my hoodie in iron mesh will resist him in a light I opens the way to him does not turn you around the future opens his day jar to make the right choice I climb on board the ship today the big jumpThe big jump
74538,bisz,indygo,https://open.spotify.com/track/2XVzSdrpRpJN2PWeifnDet,pl,zwrotka 1 urodziłeś się królem pokaż wszystkim koronę kłów berło pięści wznieś jak księżyc nad morzem głów twój tron nosisz ze sobą ciągle na spodzie stóp nie szata czyni króla masz na sobie tylko spojrzeń rój niebo jest twoim dachem orszakiem ptaki o świcie błazny masy dla ciebie jak ty dla nich bo żyjesz poza ich pogonią i za czym za słoną cenę zapłaci każdy z nich gdy obudzi się za zasłoną nie ma nic poza spokojnym oddechem pierś bóstwa rozszerza się wszechświatem potem zapada się w punkt aby na nowo wziąć wdech i spokojnie się wzwyż wznieść cichy proces żaden wydumany big bang między ziemią a niebem jest człowiek z głową w chmurach stąpając twardo przewodzę przez moje życie tę siłę która czyni ciężkość lekką baranek który jest lwem jestem piękną bestią zmieszaj najjaśniejszy błękit z jak najgłębszą czernią refren wszyscy którzy wrócili z biegunów zawsze będą ze mną weź najciemniejszą czerń zmieszaj ją z jasnoniebieskim wyzwolony od zwycięstw i klęsk obserwuj ich zmierzch w indygo indygo indygo indygo indygo indygo indygo bridge gdy już nie wiesz dokąd iść słuchaj krwi która krąży w tobie daj się ponieść daj się ponieść w kolorze twojej krwi tkwi symbol i odpowiedź daj się ponieść daj się ponieść zwrotka 2 masz w sobie coś co możesz pokazać tylko wybranym wciąż jest nas zbyt mało by doprowadzić do zmiany lecz każdy z nas ma moc by zacząć powoli wsączać łzy w serce skały aby poruszyć monolit wiemy że możemy sprawić aby świat był lepszy i kto musi wziąć na barki trud nowi architekci my bo tu miejsca dla nas brak wciąż i gdy tylko chcesz otworzyć się słyszysz że masz się zamknąć lecz jeśli myśleli że mogą kazać ci cokolwiek tylko ze względu na twoją słabość biada im nauka dla ciebie i dla nich oby stąd wynikła nie wszyscy jeszcze się poddali chodnikowy wilk brat z otchłani bólu rozpaczy niewiary po szczyty nieba zrozumienia wolności i chwały stratowany fortuny kołem wiem jedno że to odległość pomiędzy górą a dołem jest pełnią refren outro gdy już nie wiesz dokąd iść słuchaj krwi która krąży w tobie daj się ponieść daj się ponieść w kolorze twojej krwi tkwi symbol i odpowiedź włóż koronę włóż koronę,"verse 1 You were born king. Show all the crown of fangs crown. ""Moon at the sea of ​​your head your throne.For them because you live beyond their pursuit a

#### 🤔💼 Otros intentos, pruebas etc...

In [ ]:
from langdetect import detect
from googletrans import Translator
import pandas as pd
import sys
import csv

# Aumentar el límite del tamaño de campo
sys.setrecursionlimit(10000)
csv.field_size_limit(10**9) 

# Configuración inicial para prueba
file_path = r'C:\Users\solan\Downloads\get_data_from_songs\data\final_idiomas.csv'  # Ruta al archivo original
output_file_path = r'C:\Users\solan\Downloads\get_data_from_songs\data\archivo_prueba_traducido.csv'  # Ruta de salida para la prueba
error_file_path = r'C:\Users\solan\Downloads\get_data_from_songs\data\errors_log_prueba.csv'  # Ruta del archivo de errores para la prueba
translator = Translator()


# Cargar el archivo completo
data = pd.read_csv(file_path, encoding='utf-8', engine='python')

# Filtrar las filas que no están en inglés
non_english_rows = []
for index, row in data.iterrows():
    if pd.isna(row['language']) or row['language'] != 'en':
        non_english_rows.append(row)
    if len(non_english_rows) >= 100:  # Seleccionar solo las primeras 100
        break

# Crear un DataFrame con las primeras 100 filas que no son inglés
test_data = pd.DataFrame(non_english_rows)

# Añadir columna para letras traducidas si no existe
if 'translated_lyrics' not in test_data.columns:
    test_data['translated_lyrics'] = None

# Crear un DataFrame vacío para almacenar errores
errors = pd.DataFrame(columns=['index', 'lyrics', 'error_message'])

# Procesar y traducir
for index, row in test_data.iterrows():
    try:
        # Detectar idioma si no está indicado
        lang = row['language'] if pd.notnull(row['language']) else detect(row['lyrics'])
        
        # Traducir solo si no está en inglés
        if lang != 'en':
            translation = translator.translate(row['lyrics'], src=lang, dest='en')
            test_data.at[index, 'translated_lyrics'] = translation.text

    except Exception as e:
        # Registrar el error en el DataFrame de errores
        errors = pd.concat([errors, pd.DataFrame({'index': [index], 
                                                  'lyrics': [row['lyrics']], 
                                                  'error_message': [str(e)]})])

# Guardar los resultados de la prueba
test_data.to_csv(output_file_path, index=False, encoding='utf-8')
errors.to_csv(error_file_path, index=False, encoding='utf-8')
print("Prueba completada. Resultados y errores guardados.")



In [ ]:
import pandas as pd

# 📂 Ruta del archivo original
file_path = r"C:\Users\solan\Downloads\get_data_from_songs\src\functions b\df_lyrics_faltan_traduc.csv"
output_filtered_path = r"C:\Users\solan\Downloads\get_data_from_songs\data\filtrado_para_traduccion.csv"

# 🔹 Cargar el archivo original
df = pd.read_csv(file_path, low_memory=False)

# 🔹 Filtrar: donde `language` no sea "en" y `translated_lyrics` esté vacío o NaN
df_filtered = df[(df['language'] != "en") & (df['translated_lyrics'].isna())]

# 🔹 Guardar el nuevo CSV para usar en la traducción
df_filtered.to_csv(output_filtered_path, index=False, encoding="utf-8")

print(f"✅ Archivo filtrado guardado en: {output_filtered_path}")


✅ Archivo filtrado guardado en: C:\Users\solan\Downloads\get_data_from_songs\data\filtrado_para_traduccion.csv


In [ ]:
import sys
import csv
import pandas as pd
from langdetect import detect
from googletrans import Translator
import re
from tqdm import tqdm  # Importar tqdm para barra de progreso

# Aumentar el límite del tamaño de campo
sys.setrecursionlimit(10000)
csv.field_size_limit(10**9)

# Configuración inicial
file_path = r'C:\Users\solan\Downloads\get_data_from_songs\data\filtrado_para_traduccion.csvv'  # Ruta al archivo original
output_file_path = r'C:\Users\solan\Downloads\get_data_from_songs\data\df_scrap_traducido.csv'  # Ruta de salida
error_file_path = r'C:\Users\solan\Downloads\get_data_from_songs\data\errors_log_traduccion.csv'  # Archivo de errores
batch_size = 1000  # Guardar cada 1,000 filas procesadas
translator = Translator()

# Función para limpiar texto
def clean_text(text):
    if pd.isna(text):
        return ""
    # Reemplazar saltos de línea y espacios múltiples por un solo espacio
    text = re.sub(r'\s+', ' ', text.replace('\n', ' ').strip())
    return text

# Cargar datos
data = pd.read_csv(file_path, encoding='utf-8', engine='python')

# Añadir columna para letras traducidas si no existe
if 'translated_lyrics' not in data.columns:
    data['translated_lyrics'] = None

# Crear un DataFrame vacío para almacenar errores
errors = pd.DataFrame(columns=['index', 'lyrics', 'error_message'])

# Procesar y traducir con barra de progreso
for index, row in tqdm(data.iterrows(), total=data.shape[0], desc="Procesando letras de canciones"):
    try:
        # Verificar si ya existe una traducción
        if pd.notnull(row['translated_lyrics']):
            continue

        # Limpiar la letra antes de procesar
        lyrics_cleaned = clean_text(row['lyrics'])

        # Detectar idioma si no está indicado
        lang = row['language'] if pd.notnull(row['language']) else detect(lyrics_cleaned)

        # Traducir solo si no está en inglés
        if lang != 'en' and lyrics_cleaned:
            translation = translator.translate(lyrics_cleaned, src=lang, dest='en')
            data.at[index, 'translated_lyrics'] = translation.text

        # Guardar cada batch de filas procesadas
        if index % batch_size == 0:
            data.to_csv(output_file_path, index=False, encoding='utf-8')
            errors.to_csv(error_file_path, index=False, encoding='utf-8')

    except Exception as e:
        # Registrar el error en el DataFrame de errores
        errors = pd.concat([errors, pd.DataFrame({'index': [index], 
                                                  'lyrics': [row['lyrics']], 
                                                  'error_message': [str(e)]})])

# Guardar archivo final
data.to_csv(output_file_path, index=False, encoding='utf-8')
errors.to_csv(error_file_path, index=False, encoding='utf-8')
print("Proceso completado y archivo final guardado.")


Opcion deep trasnlator

In [ ]:
from deep_translator import GoogleTranslator

# Crear un traductor de español a inglés
translator = GoogleTranslator(source="auto", target="en")

# Traducir una frase de prueba
translation = translator.translate("Hola mundo")
print(translation)  # Output: "Hello world"



Hello world


In [ ]:
translated_data = translated_data.drop_duplicates(subset=['recording_id'])

data = data.merge(translated_data[['recording_id', 'translated_lyrics']], on='recording_id', how='left', suffixes=('', '_prev'))
data['translated_lyrics'] = data['translated_lyrics'].combine_first(data['translated_lyrics_prev'])
data.drop(columns=['translated_lyrics_prev'], inplace=True)


In [ ]:
if os.path.exists(output_file_path):
    translated_data = pd.read_csv(output_file_path, encoding='utf-8', engine='python', on_bad_lines="skip")
    print("📌 Columnas en traducidas.csv:", translated_data.columns)


NameError: name 'os' is not defined

otro modelo - no

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
from langdetect import detect
import pandas as pd

# Configuración del modelo MarianMT
model_name = "Helsinki-NLP/opus-mt-es-en"  # Cambia el modelo según el idioma (ej. `es-en`, `fr-en`)
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Función para traducir usando MarianMT
def translate_marian(texts, source_lang="es", target_lang="en"):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    translated = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in translated]

# Cargar datos del archivo
file_path = r'C:\ruta\a\final_idiomas.csv'  # Ruta del archivo original
output_file_path = r'C:\ruta\a\final_idiomas_traducido_marian.csv'  # Archivo de salida
data = pd.read_csv(file_path, encoding='utf-8')

# Añadir columna para letras traducidas si no existe
if 'translated_lyrics' not in data.columns:
    data['translated_lyrics'] = None

# Procesar y traducir
for index, row in data.iterrows():
    try:
        # Verificar si ya existe una traducción
        if pd.notnull(row['translated_lyrics']):
            continue
        
        # Detectar idioma si no está especificado
        lyrics = row['lyrics']
        lang = row['language'] if pd.notnull(row['language']) else detect(lyrics)

        # Traducir solo si el idioma no es inglés
        if lang != 'en' and pd.notnull(lyrics):
            translated_text = translate_marian([lyrics])[0]
            data.at[index, 'translated_lyrics'] = translated_text

    except Exception as e:
        print(f"Error al traducir en la fila {index}: {e}")

# Guardar archivo final con traducciones
data.to_csv(output_file_path, index=False, encoding='utf-8')
print(f"Traducciones completadas. Archivo guardado en {output_file_path}.")


In [ ]:
pd.set_option('display.max_colwidth', None)

NameError: name 'pd' is not defined

In [4]:
import pandas as pd
import csv

# 📂 Rutas de los archivos
final_url_path = r"C:\Users\solan\MoodTune\data\procesando\df_80-200_p2_last.csv"
translated_lyrics_path = r"C:\Users\solan\MoodTune\data\procesando\df_80-200_p2_last_traducidas.csv"
output_path = r"C:\Users\solan\MoodTune\data\procesando\df_80-200_p2_traducidas.csv"

# 1️⃣ Cargar los archivos CSV
df_final = pd.read_csv(final_url_path, low_memory=False)
df_translations = pd.read_csv(translated_lyrics_path, low_memory=False)

# 2️⃣ Verificar que 'recording_id' existe en ambos
if 'recording_id' not in df_final.columns or 'recording_id' not in df_translations.columns:
    raise KeyError("❌ ERROR: 'recording_id' no se encuentra en ambos datasets.")

# 3️⃣ Seleccionar solo la columna 'recording_id' y 'translated_lyrics' para fusionar
df_translations = df_translations[['recording_id', 'translated_lyrics']]

# 4️⃣ Reemplazar saltos de línea por un espacio simple y limpiar espacios extra
df_translations['translated_lyrics'] = df_translations['translated_lyrics'].astype(str).apply(lambda x: " ".join(x.split()))

# 5️⃣ Hacer el merge por 'recording_id'
df_merged = df_final.merge(df_translations, on='recording_id', how='left')

# 6️⃣ Guardar el archivo final con formato correcto
df_merged.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL, lineterminator='\n')

print(f"✅ Archivo final guardado con las traducciones en: {output_path}")


✅ Archivo final guardado con las traducciones en: C:\Users\solan\MoodTune\data\procesando\df_80-200_p2_traducidas.csv


In [ ]:
df_non_english.head()